# OpsAssist - Week 9 API Notebook
**Applied GenAI & Agentic AI Engineering Course · Week 9**

OpsAssist is a tool-using ops agent built in raw Python. This notebook drives every endpoint two ways - `%%cmd` curl and Python `requests` - so you can see the exact wire format before the browser UI abstracts it.

Each endpoint is shown two ways:
- **cURL (Windows cmd)** - `%%cmd` cell magic, Windows double-quote syntax, single-line only
- **Python** - `requests` library, works everywhere

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` filled in with `OPENAI_API_KEY` (see `.env.example`)
3. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.

In [ ]:
# Setup -- run this cell first
import requests, json

BASE = 'http://localhost:8000'

# ASCII only -- no em dashes or Unicode (breaks Windows cmd curl).
DEMO_NOTES = 'auth-service latency has spiked in the last hour. p95 is over 400ms. Investigate and propose a remediation.'

# Quick-fill variants -- match the four runbooks in tools.py
DEMO = {
    'auth':   'auth-service latency has spiked in the last hour. p95 is over 400ms. Investigate and propose a remediation.',
    'api':    'Error rate on the API service is elevated over the last 5 minutes. Diagnose and recommend next steps.',
    'queue':  'Worker queue backlog is growing. Investigate the cause and propose a scale action.',
    'db':     'Database connection pool is exhausted. Identify which client is causing the leak and propose a fix.',
}

# Fixed task_id for the workflow-state resume demo (Section 3)
TASK_ID = 'demo-task-001'

print('Setup complete.')
print('BASE:', BASE)
print('DEMO_NOTES:', DEMO_NOTES[:60], '...')

---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [ ]:
%%cmd
curl -s http://localhost:8000/health

In [ ]:
# Health check -- Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 2 · Run Agent - `POST /agent/run`

The core endpoint. Send a free-form incident description; the agent loop runs at most `MAX_ITERS` (default 10) iterations, calling tools, injecting results, and checking six stop conditions. Returns a final response plus a structured trace the eval harness consumes.

**Key concept:** This is the four-step loop from V1 running in Python - `call_with_tools` → `execute_tool` → inject result → stop-check - repeated until `end_turn`, `max_iters`, `token_budget`, `fatal_tool_error`, `consent_revoked`, or `timeout`.

Request body:
```json
{ "user_input": "...", "user_id": "alice", "task_id": null }
```

Response shape:
```json
{
  "final_response": "Based on my analysis...",
  "iter_count": 3,
  "stop_reason": "end_turn",
  "trace": [
    {
      "iter": 0,
      "model_request_tokens": 210,
      "model_response_tokens": 85,
      "tool_calls": [{"name": "get_runbook", "input": {"topic": "auth-service-latency"}, "id": "call_..."}],
      "tool_results": [{"tool_call_id": "call_...", "is_error": false, "content_preview": "..."}],
      "elapsed_ms": 1240
    }
  ]
}
```

> **Requires:** `OPENAI_API_KEY` in `.env`. Restart uvicorn after editing `.env`.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/agent/run -H "Content-Type: application/json" -d "{\"user_input\": \"auth-service latency has spiked. p95 over 400ms. Investigate and propose a remediation.\", \"user_id\": \"demo\"}"

In [ ]:
# Run Agent -- Python
r = requests.post(f'{BASE}/agent/run', json={
    'user_input': DEMO_NOTES,
    'user_id':    'demo',
})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('--- Final Response ---')
    print(data['final_response'])
    print()
    print(f'Stop reason : {data["stop_reason"]}')
    print(f'Iterations  : {data["iter_count"]}')
    print(f'Trace steps : {len(data["trace"])}')
    print()
    print('--- Tool calls per iteration ---')
    for it in data['trace']:
        tools = [tc['name'] for tc in it['tool_calls']] or ['(no tools -- final turn)']
        tok   = it['model_request_tokens'] + it['model_response_tokens']
        print(f'  iter {it["iter"]}: {tools}  ({tok} tok, {it["elapsed_ms"]}ms)')

---
## 3 · Resume with Task ID - `POST /agent/run` (workflow state)

OpsAssist supports multi-turn sessions via an explicit `task_id`. When you supply the same `task_id` on a second call, the server reads the last workflow checkpoint and restores the conversation messages, iteration count, and token usage before running the new loop. This lets an engineer follow up on an earlier diagnosis without repeating the full investigation.

**Key concept (V3 state architecture):** `WorkflowState` uses Redis in production (or a process-local dict in dev). `write_checkpoint()` is called after every successful iteration; the next call to `/agent/run` with the same `task_id` reads it via `read_checkpoint()` and picks up where it left off.

> **Note:** In local dev (no `REDIS_URL` set), the checkpoint lives in an in-process dict - it survives within the same uvicorn process but resets on restart. In production (Redis), it survives across restarts with a 24h TTL.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/agent/run -H "Content-Type: application/json" -d "{\"user_input\": \"Re-check auth-service status after credential rotation.\", \"task_id\": \"demo-task-001\", \"user_id\": \"demo\"}"

In [ ]:
# Step 1: first run -- establish workflow state under TASK_ID
r1 = requests.post(f'{BASE}/agent/run', json={
    'user_input': DEMO['auth'],
    'user_id':    'demo',
    'task_id':    TASK_ID,
})
d1 = r1.json()
print('=== First run ===')
print(f'task_id    : {TASK_ID}')
print(f'stop_reason: {d1["stop_reason"]}')
print(f'iter_count : {d1["iter_count"]}')
print()

# Step 2: follow-up run -- same task_id, different user_input
r2 = requests.post(f'{BASE}/agent/run', json={
    'user_input': 'Re-check auth-service status after credential rotation.',
    'user_id':    'demo',
    'task_id':    TASK_ID,
})
d2 = r2.json()
print('=== Resume run (same task_id) ===')
print(f'task_id    : {TASK_ID}')
print(f'stop_reason: {d2["stop_reason"]}')
print(f'iter_count : {d2["iter_count"]}')
print()
print('--- Follow-up response ---')
print(d2['final_response'])

---
## 4 · Full Raw Response
Shows the complete JSON as returned by the API - useful for debugging the trace structure.

In [ ]:
# Full raw response dump -- run a fresh call with the api-error scenario
r = requests.post(f'{BASE}/agent/run', json={
    'user_input': DEMO['api'],
    'user_id':    'demo',
})
print(json.dumps(r.json(), indent=2))

---
## 5 · Failure Mode - Missing Required Field (422)
Pydantic validates the request body before the agent loop is even entered - no API call is made, no tokens are spent.

`user_input` is a required field (no default). Omitting it returns **422 Unprocessable Entity** immediately.

> This failure pattern is identical across all weeks. The field name and endpoint change; the 422 shape does not.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/agent/run -H "Content-Type: application/json" -d "{\"user_id\": \"demo\"}"

In [ ]:
# Failure: missing user_input -- Python
r = requests.post(f'{BASE}/agent/run', json={'user_id': 'demo'})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects missing required field, no loop entered)')
body = r.json()
# FastAPI 422 detail is a list of validation errors
for err in body.get('detail', []):
    print(f'  field: {err.get("loc")}  msg: {err.get("msg")}')

---
## 6 · Failure Mode - Agent Cannot Help (Unknown Issue)

OpsAssist's tool schemas use `Literal[...]` types that constrain inputs to the known runbooks and service names. When asked about a topic outside that set, the model receives no matching runbook and must respond without evidence.

**What you see:** `stop_reason: "end_turn"` with a final response explaining it cannot help - no tool calls fire, no PagerDuty incident is created.

**The lesson (V1 - stopping policy):** the system prompt says *"If you cannot answer with the tools available, return a final response that says so and stop."* A well-bounded stopping policy prevents the agent from spinning or hallucinating tool names that don't exist.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/agent/run -H "Content-Type: application/json" -d "{\"user_input\": \"Investigate why the Kubernetes node pool is scaling unexpectedly.\", \"user_id\": \"demo\"}"

In [ ]:
# Failure: out-of-scope issue -- Python
# The agent has no runbook or metric for Kubernetes node scaling.
# It should stop gracefully rather than hallucinate a tool call.
r = requests.post(f'{BASE}/agent/run', json={
    'user_input': 'Investigate why the Kubernetes node pool is scaling unexpectedly.',
    'user_id':    'demo',
})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print(f'Stop reason : {data["stop_reason"]}  (expected end_turn -- agent stops without tools)')
    print(f'Iterations  : {data["iter_count"]}')
    tool_calls = sum(len(it['tool_calls']) for it in data['trace'])
    print(f'Tool calls  : {tool_calls}  (expected 0 or very few)')
    print()
    print('--- Final response ---')
    print(data['final_response'])

---
## 7 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))